# ForestGuard: small satellite sample
CPU cloud only, no paid services or new packages. Kaggle Internet must be enabled. The area is provisional and does not represent the official Joga beat. No model training or accuracy claims.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import platform
import shutil
import zipfile
from datetime import datetime, timezone

if Path("/kaggle/working").is_dir():
    provider = "Kaggle"
    output_dir = Path("/kaggle/working/forestguard")
elif Path("/content").is_dir():
    try:
        import google.colab
    except ImportError as error:
        raise RuntimeError("Use a hosted Colab runtime, not a local runtime.") from error
    provider = "Colab"
    output_dir = Path("/content/forestguard")
else:
    raise RuntimeError("Use a hosted Kaggle or Colab runtime. This check intentionally stops on your laptop.")

output_dir.mkdir(parents=True, exist_ok=True)
print(f"Remote runtime: {provider}; output directory: {output_dir}")


## Read and check a small crop
Four spectral bands provide measurements. The scene classification layer excludes cloud, shadow and uncertain pixels; it does not supply forest labels. Scale and offset come from the public STAC metadata. RGB brightness is adjusted for display only.


In [ ]:
META = {'id': 'S2C_43QFE_20250329_0_L2A', 'datetime': '2025-03-29T05:43:09.925000Z', 'assets': {'red': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B04.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Red - 10m', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'green': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B03.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Green - 10m', 'eo:bands': [{'name': 'B03', 'common_name': 'green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'blue': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B02.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Blue - 10m', 'eo:bands': [{'name': 'B02', 'common_name': 'blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B08.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'NIR 1 - 10m', 'eo:bands': [{'name': 'B08', 'common_name': 'nir', 'center_wavelength': 0.842, 'full_width_half_max': 0.145}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'scl': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/SCL.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Scene classification map (SCL)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 20}], 'roles': ['data']}}}
ITEM = {'type': 'Feature', 'stac_version': '1.0.0', 'stac_extensions': ['https://stac-extensions.github.io/view/v1.0.0/schema.json', 'https://stac-extensions.github.io/eo/v1.1.0/schema.json', 'https://stac-extensions.github.io/sentinel-2/v1.0.0/schema.json', 'https://stac-extensions.github.io/processing/v1.1.0/schema.json', 'https://stac-extensions.github.io/projection/v1.1.0/schema.json', 'https://stac-extensions.github.io/grid/v1.1.0/schema.json', 'https://stac-extensions.github.io/raster/v1.1.0/schema.json', 'https://stac-extensions.github.io/mgrs/v1.0.0/schema.json'], 'id': 'S2C_43QFE_20250329_0_L2A', 'geometry': {'type': 'Polygon', 'coordinates': [[[75.972947682903, 22.604080062666913], [75.96616817389219, 21.612258586164263], [76.61484171029622, 21.60722594856719], [76.86830953038077, 22.596156676716596], [75.972947682903, 22.604080062666913]]]}, 'bbox': [75.966168, 21.607226, 76.86831, 22.60408], 'properties': {'created': '2025-03-29T12:16:30.187Z', 'platform': 'sentinel-2c', 'constellation': 'sentinel-2', 'instruments': ['msi'], 'eo:cloud_cover': 0.005916, 'proj:epsg': 32643, 'proj:centroid': {'lat': 22.13131, 'lon': 76.35898}, 'mgrs:utm_zone': 43, 'mgrs:latitude_band': 'Q', 'mgrs:grid_square': 'FE', 'grid:code': 'MGRS-43QFE', 'view:azimuth': 285.64465901301526, 'view:incidence_angle': 8.625167743986273, 'view:sun_azimuth': 132.791834407871, 'view:sun_elevation': 63.880865232279305, 's2:tile_id': 'S2C_OPER_MSI_L2A_TL_2CPS_20250329T105014_A002935_T43QFE_N05.11', 's2:degraded_msi_data_percentage': 0.0154, 's2:nodata_pixel_percentage': 27.369893, 's2:saturated_defective_pixel_percentage': 0, 's2:cloud_shadow_percentage': 0, 's2:vegetation_percentage': 3.090336, 's2:not_vegetated_percentage': 88.784051, 's2:water_percentage': 6.471512, 's2:unclassified_percentage': 1.421178, 's2:medium_proba_clouds_percentage': 0.005633, 's2:high_proba_clouds_percentage': 0, 's2:thin_cirrus_percentage': 0.000283, 's2:snow_ice_percentage': 0, 's2:product_type': 'S2MSI2A', 's2:processing_baseline': '05.11', 's2:product_uri': 'S2C_MSIL2A_20250329T052841_N0511_R105_T43QFE_20250329T105014.SAFE', 's2:generation_time': '2025-03-29T10:50:14.000000Z', 's2:datatake_id': 'GS2C_20250329T052841_002935_N05.11', 's2:datatake_type': 'INS-NOBS', 's2:datastrip_id': 'S2C_OPER_MSI_L2A_DS_2CPS_20250329T105014_S20250329T053314_N05.11', 's2:reflectance_conversion_factor': 1.00587010401578, 'datetime': '2025-03-29T05:43:09.925000Z', 's2:sequence': '0', 'earthsearch:s3_path': 's3://sentinel-cogs/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A', 'earthsearch:payload_id': 'roda-sentinel2/workflow-sentinel2-to-stac/97199fd52ba38aec83c7ad73e5086fc2', 'earthsearch:boa_offset_applied': True, 'processing:software': {'sentinel2-to-stac': '2025.03.06'}, 'updated': '2025-03-29T12:16:30.187Z'}, 'links': [{'rel': 'self', 'type': 'application/geo+json', 'href': 'https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/S2C_43QFE_20250329_0_L2A'}, {'rel': 'canonical', 'href': 's3://sentinel-cogs/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/S2C_43QFE_20250329_0_L2A.json', 'type': 'application/json'}, {'rel': 'license', 'href': 'https://sentinel.esa.int/documents/247904/690755/Sentinel_Data_Legal_Notice'}, {'rel': 'derived_from', 'href': 'https://earth-search.aws.element84.com/v1/collections/sentinel-2-l1c/items/S2C_43QFE_20250329_0_L1C', 'type': 'application/geo+json'}, {'rel': 'parent', 'type': 'application/json', 'href': 'https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a'}, {'rel': 'collection', 'type': 'application/json', 'href': 'https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a'}, {'rel': 'root', 'type': 'application/json', 'href': 'https://earth-search.aws.element84.com/v1'}, {'rel': 'thumbnail', 'href': 'https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/S2C_43QFE_20250329_0_L2A/thumbnail'}], 'assets': {'aot': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/AOT.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Aerosol optical thickness (AOT)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.001, 'offset': 0}], 'roles': ['data']}, 'blue': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B02.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Blue - 10m', 'eo:bands': [{'name': 'B02', 'common_name': 'blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'cloud': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/qi/CLD_20m.jp2', 'type': 'image/jp2', 'title': 'Cloud Probabilities', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 20}], 'roles': ['data', 'cloud']}, 'coastal': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B01.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Coastal - 60m', 'eo:bands': [{'name': 'B01', 'common_name': 'coastal', 'center_wavelength': 0.443, 'full_width_half_max': 0.027}], 'gsd': 60, 'proj:shape': [1830, 1830], 'proj:transform': [60, 0, 600000, 0, -60, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 60, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'granule_metadata': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/granule_metadata.xml', 'type': 'application/xml', 'roles': ['metadata']}, 'green': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B03.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Green - 10m', 'eo:bands': [{'name': 'B03', 'common_name': 'green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B08.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'NIR 1 - 10m', 'eo:bands': [{'name': 'B08', 'common_name': 'nir', 'center_wavelength': 0.842, 'full_width_half_max': 0.145}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir08': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B8A.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'NIR 2 - 20m', 'eo:bands': [{'name': 'B8A', 'common_name': 'nir08', 'center_wavelength': 0.865, 'full_width_half_max': 0.033}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir09': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B09.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'NIR 3 - 60m', 'eo:bands': [{'name': 'B09', 'common_name': 'nir09', 'center_wavelength': 0.945, 'full_width_half_max': 0.026}], 'gsd': 60, 'proj:shape': [1830, 1830], 'proj:transform': [60, 0, 600000, 0, -60, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 60, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'product_metadata': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/product_metadata.xml', 'type': 'application/xml', 'roles': ['metadata']}, 'red': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B04.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Red - 10m', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge1': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B05.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Red Edge 1 - 20m', 'eo:bands': [{'name': 'B05', 'common_name': 'rededge', 'center_wavelength': 0.704, 'full_width_half_max': 0.019}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge2': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B06.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Red Edge 2 - 20m', 'eo:bands': [{'name': 'B06', 'common_name': 'rededge', 'center_wavelength': 0.74, 'full_width_half_max': 0.018}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge3': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B07.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Red Edge 3 - 20m', 'eo:bands': [{'name': 'B07', 'common_name': 'rededge', 'center_wavelength': 0.783, 'full_width_half_max': 0.028}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'scl': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/SCL.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Scene classification map (SCL)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 20}], 'roles': ['data']}, 'snow': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/qi/SNW_20m.jp2', 'type': 'image/jp2', 'title': 'Snow Probabilities', 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 20}], 'roles': ['data', 'snow-ice']}, 'swir16': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B11.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'SWIR 1.6μm - 20m', 'eo:bands': [{'name': 'B11', 'common_name': 'swir16', 'center_wavelength': 1.61, 'full_width_half_max': 0.143}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'swir22': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/B12.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'SWIR 2.2μm - 20m', 'eo:bands': [{'name': 'B12', 'common_name': 'swir22', 'center_wavelength': 2.19, 'full_width_half_max': 0.242}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'tileinfo_metadata': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/tileinfo_metadata.json', 'type': 'application/json', 'roles': ['metadata']}, 'visual': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/TCI.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'True color image', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}, {'name': 'B03', 'common_name': 'green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}, {'name': 'B02', 'common_name': 'blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}, {'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}, {'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'roles': ['visual']}, 'wvp': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/WVP.tif', 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'title': 'Water Vapour (WVP)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'unit': 'cm', 'scale': 0.001, 'offset': 0}], 'roles': ['data']}, 'thumbnail': {'href': 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/Q/FE/2025/3/S2C_43QFE_20250329_0_L2A/preview.jpg', 'type': 'image/jpeg', 'title': 'Thumbnail of preview image', 'roles': ['thumbnail']}, 'aot-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/AOT.jp2', 'type': 'image/jp2', 'title': 'Aerosol optical thickness (AOT)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.001, 'offset': 0}], 'roles': ['data']}, 'blue-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R10m/B02.jp2', 'type': 'image/jp2', 'title': 'Blue - 10m', 'eo:bands': [{'name': 'B02', 'common_name': 'blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'coastal-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R60m/B01.jp2', 'type': 'image/jp2', 'title': 'Coastal - 60m', 'eo:bands': [{'name': 'B01', 'common_name': 'coastal', 'center_wavelength': 0.443, 'full_width_half_max': 0.027}], 'gsd': 60, 'proj:shape': [1830, 1830], 'proj:transform': [60, 0, 600000, 0, -60, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 60, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'green-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R10m/B03.jp2', 'type': 'image/jp2', 'title': 'Green - 10m', 'eo:bands': [{'name': 'B03', 'common_name': 'green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R10m/B08.jp2', 'type': 'image/jp2', 'title': 'NIR 1 - 10m', 'eo:bands': [{'name': 'B08', 'common_name': 'nir', 'center_wavelength': 0.842, 'full_width_half_max': 0.145}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir08-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B8A.jp2', 'type': 'image/jp2', 'title': 'NIR 2 - 20m', 'eo:bands': [{'name': 'B8A', 'common_name': 'nir08', 'center_wavelength': 0.865, 'full_width_half_max': 0.033}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'nir09-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R60m/B09.jp2', 'type': 'image/jp2', 'title': 'NIR 3 - 60m', 'eo:bands': [{'name': 'B09', 'common_name': 'nir09', 'center_wavelength': 0.945, 'full_width_half_max': 0.026}], 'gsd': 60, 'proj:shape': [1830, 1830], 'proj:transform': [60, 0, 600000, 0, -60, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 60, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'red-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R10m/B04.jp2', 'type': 'image/jp2', 'title': 'Red - 10m', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 10, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge1-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B05.jp2', 'type': 'image/jp2', 'title': 'Red Edge 1 - 20m', 'eo:bands': [{'name': 'B05', 'common_name': 'rededge', 'center_wavelength': 0.704, 'full_width_half_max': 0.019}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge2-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B06.jp2', 'type': 'image/jp2', 'title': 'Red Edge 2 - 20m', 'eo:bands': [{'name': 'B06', 'common_name': 'rededge', 'center_wavelength': 0.74, 'full_width_half_max': 0.018}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'rededge3-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B07.jp2', 'type': 'image/jp2', 'title': 'Red Edge 3 - 20m', 'eo:bands': [{'name': 'B07', 'common_name': 'rededge', 'center_wavelength': 0.783, 'full_width_half_max': 0.028}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'scl-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/SCL.jp2', 'type': 'image/jp2', 'title': 'Scene classification map (SCL)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 20}], 'roles': ['data']}, 'swir16-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B11.jp2', 'type': 'image/jp2', 'title': 'SWIR 1.6μm - 20m', 'eo:bands': [{'name': 'B11', 'common_name': 'swir16', 'center_wavelength': 1.61, 'full_width_half_max': 0.143}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'swir22-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/B12.jp2', 'type': 'image/jp2', 'title': 'SWIR 2.2μm - 20m', 'eo:bands': [{'name': 'B12', 'common_name': 'swir22', 'center_wavelength': 2.19, 'full_width_half_max': 0.242}], 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'scale': 0.0001, 'offset': -0.1}], 'roles': ['data', 'reflectance']}, 'visual-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R10m/TCI.jp2', 'type': 'image/jp2', 'title': 'True color image', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}, {'name': 'B03', 'common_name': 'green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}, {'name': 'B02', 'common_name': 'blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}], 'raster:bands': [{'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}, {'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}, {'nodata': 0, 'data_type': 'uint8', 'spatial_resolution': 10}], 'gsd': 10, 'proj:shape': [10980, 10980], 'proj:transform': [10, 0, 600000, 0, -10, 2500020], 'roles': ['visual']}, 'wvp-jp2': {'href': 's3://sentinel-s2-l2a/tiles/43/Q/FE/2025/3/29/0/R20m/WVP.jp2', 'type': 'image/jp2', 'title': 'Water Vapour (WVP)', 'gsd': 20, 'proj:shape': [5490, 5490], 'proj:transform': [20, 0, 600000, 0, -20, 2500020], 'raster:bands': [{'nodata': 0, 'data_type': 'uint16', 'spatial_resolution': 20, 'unit': 'cm', 'scale': 0.001, 'offset': 0}], 'roles': ['data']}}, 'collection': 'sentinel-2-l2a'}
import math, time, zipfile, hashlib
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds, transform as project_points, Resampling
from rasterio.vrt import WarpedVRT
from PIL import Image

started = time.monotonic()
BBOX = [76.785, 22.413, 76.805, 22.433]
OUT = Path('/kaggle/working/forestguard_sample' if Path('/kaggle/working').is_dir() else '/content/forestguard_sample')
OUT.mkdir(exist_ok=True)
# Read only intersecting COG blocks; no full-scene download or AWS credentials.
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif', GDAL_HTTP_TIMEOUT='60', GDAL_CACHEMAX=64*1024*1024):
    with rasterio.open(META['assets']['red']['href']) as src:
        projected = transform_bounds('EPSG:4326', src.crs, *BBOX, densify_pts=21)
        w = from_bounds(*projected, transform=src.transform)
        left, top = math.floor(w.col_off), math.floor(w.row_off)
        right, bottom = math.ceil(w.col_off+w.width), math.ceil(w.row_off+w.height)
        window = Window(left, top, right-left, bottom-top)
        assert 0 <= left < right <= src.width and 0 <= top < bottom <= src.height, 'Crop outside scene'
        assert max(window.width, window.height) <= 512, 'Preview size limit'
        crs, transform = src.crs, src.window_transform(window)
        source_transform = src.transform
        height, width = int(window.height), int(window.width)
    bands, masks = [], []
    for key in ['blue','green','red','nir']:
        asset = META['assets'][key]
        calibration = asset['raster:bands'][0]
        with rasterio.open(asset['href']) as src:
            assert src.crs == crs and src.transform == source_transform and src.res == (10.0,10.0), 'Band grid mismatch'
            raw = src.read(1, window=window, masked=True)
        masks.append(~np.ma.getmaskarray(raw))
        bands.append(raw.filled(0).astype('float32') * calibration['scale'] + calibration['offset'])
    reflectance = np.stack(bands)
    with rasterio.open(META['assets']['scl']['href']) as src:
        with WarpedVRT(src, crs=crs, transform=transform, width=width, height=height, resampling=Resampling.nearest) as vrt:
            scl = vrt.read(1)
# Pixel centers define the research-box denominator; rounded window margins are excluded.
rows, cols = np.indices((height,width))
x = transform.c + (cols + .5) * transform.a
y = transform.f + (rows + .5) * transform.e
lon, lat = project_points(crs, 'EPSG:4326', x.ravel(), y.ravel())
lon, lat = np.array(lon).reshape(height,width), np.array(lat).reshape(height,width)
inside = (lon >= BBOX[0]) & (lon <= BBOX[2]) & (lat >= BBOX[1]) & (lat <= BBOX[3])
assert inside.any(), 'No research-box pixel centers'
valid = inside & np.logical_and.reduce(masks) & np.isfinite(reflectance).all(axis=0) & np.isin(scl, [4,5,6])
assert valid.any(), 'No usable vegetation/bare/water pixels in crop'
reflectance[:,~valid] = -9999
profile = dict(driver='GTiff', crs=crs, transform=transform, width=width, height=height, compress='deflate')
with rasterio.open(OUT/'reflectance.tif','w',**profile,count=4,dtype='float32',nodata=-9999) as dst:
    dst.write(reflectance)
    dst.descriptions = ('B02 blue','B03 green','B04 red','B08 near infrared')
for name, array in [('scene_classes.tif',scl),('usable_mask.tif',valid.astype('uint8')),('study_mask.tif',inside.astype('uint8'))]:
    with rasterio.open(OUT/name,'w',**profile,count=1,dtype='uint8') as dst:
        dst.write(array,1)
rgb = (np.clip(reflectance[[2,1,0]] / 0.3,0,1)*255).astype('uint8').transpose(1,2,0)
rgba = np.dstack([rgb,valid.astype('uint8')*255])
Image.fromarray(rgba).save(OUT/'preview.png')
with rasterio.open(OUT/'reflectance.tif') as check:
    assert check.count == 4 and check.crs == crs and check.transform == transform
    assert check.shape == (height,width) and check.res == (10.0,10.0)
    assert np.array_equal(check.read(1)!=-9999, valid)
classes, counts = np.unique(scl[inside],return_counts=True)
report = dict(scene_id=META['id'],date=META['datetime'],bbox_lon_lat=BBOX,area_status='provisional research crop; not a beat boundary',
    shape=[height,width],crs=str(crs),resolution_m=10,band_order=['B02','B03','B04','B08'],scale_offset_applied=True,
    transform=list(transform)[:6],study_pixels=int(inside.sum()),usable_pixels=int(valid.sum()),
    usable_fraction=float(valid.sum()/inside.sum()),scene_class_counts_inside_study={str(int(k)):int(v) for k,v in zip(classes,counts)},
    mask_rule='SCL 4,5,6 and all bands valid; SCL is a quality filter, not forest ground truth',
    model_trained=False,forest_labels_available=False,processing_seconds=round(time.monotonic()-started,2),source=META,
    schema_version=2,generated_at_utc=datetime.now(timezone.utc).isoformat(),
    preprocessing={'reflectance_formula':'raw * asset scale + asset offset; once only',
        'scl_resampling':'nearest; native 20 m quality layer aligned to 10 m grid',
        'study_mask':'pixel centers inside the provisional EPSG:4326 research box',
        'display_only':'RGB / 0.3, clipped to 0..1; invalid pixels transparent'},
    calibration={k:META['assets'][k]['raster:bands'][0] for k in ['blue','green','red','nir']},
    calibration_status='asset metadata applied; upstream COG consistency not independently checked',
    attribution='Contains modified Copernicus Sentinel data 2025',
    license_url='https://cds.climate.copernicus.eu/licences/ec-sentinel',
    runtime={'provider':provider,'python':platform.python_version(),
        'packages':{p:metadata.version(p) for p in ['numpy','rasterio','Pillow']},
        'gdal':rasterio.__gdal_version__},
    reflectance_quantiles={name:np.quantile(reflectance[i,valid],[0,.01,.5,.99,1]).tolist()
        for i,name in enumerate(['B02','B03','B04','B08'])})
(OUT/'sample_report.json').write_text(json.dumps(report,indent=2))
(OUT/'source_stac_item.json').write_text(json.dumps(ITEM,indent=2),encoding='utf-8')
SAMPLE_FILES = ['reflectance.tif','scene_classes.tif','usable_mask.tif','study_mask.tif',
    'preview.png','sample_report.json','source_stac_item.json']
checksums = {}
for name in SAMPLE_FILES:
    file = OUT/name
    with file.open('rb') as stream:
        digest = hashlib.file_digest(stream,'sha256').hexdigest()
    checksums[name] = {'sha256':digest,'bytes':file.stat().st_size}
(OUT/'checksums.json').write_text(json.dumps(checksums,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in report.items() if k!='source'},indent=2))
from IPython.display import display
display(Image.open(OUT/'preview.png'))


## Verify and preserve the export
This checks saved raster grids, study-box coverage, quality masks, provenance and checksums. Download the ZIP after this cell succeeds. Coverage is not forest area or model accuracy.


In [ ]:
"""Verify an exported research sample offline; --rasters requires Rasterio/NumPy."""
import argparse
import hashlib
import json
import math
from pathlib import Path

FILES = {
    'reflectance.tif', 'scene_classes.tif', 'usable_mask.tif',
    'study_mask.tif', 'preview.png', 'source_stac_item.json', 'sample_report.json',
}


def verify(folder, rasters=False):
    folder = Path(folder)
    manifest = json.loads((folder / 'checksums.json').read_text(encoding='utf-8'))
    if set(manifest) != FILES:
        raise ValueError('Manifest does not list the expected sample files.')
    for name, record in manifest.items():
        path = folder / name
        if path.stat().st_size != record['bytes']:
            raise ValueError(f'File size mismatch: {name}')
        with path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != record['sha256']:
            raise ValueError(f'Checksum mismatch: {name}')
    report = json.loads((folder / 'sample_report.json').read_text(encoding='utf-8'))
    source = json.loads((folder / 'source_stac_item.json').read_text(encoding='utf-8'))
    if report['scene_id'] != source['id'] or report['date'] != source['properties']['datetime']:
        raise ValueError('Scene/date provenance mismatch.')
    if report['band_order'] != ['B02', 'B03', 'B04', 'B08']:
        raise ValueError('Unexpected band order.')
    if report['model_trained'] or report['forest_labels_available']:
        raise ValueError('A research sample cannot claim a trained model or forest labels.')
    height, width = report['shape']
    if not (0 < height <= 512 and 0 < width <= 512):
        raise ValueError('Research crop exceeds the size limit.')
    study, usable = report['study_pixels'], report['usable_pixels']
    if not (0 < usable <= study <= height * width):
        raise ValueError('Invalid coverage counts.')
    if not math.isclose(report['usable_fraction'], usable / study, abs_tol=1e-12):
        raise ValueError('Coverage denominator mismatch.')
    if sum(report['scene_class_counts_inside_study'].values()) != study:
        raise ValueError('SCL histogram count mismatch.')

    if rasters:
        import numpy as np
        import rasterio
        from rasterio.warp import transform as project_points
        with rasterio.open(folder / 'reflectance.tif') as image:
            if (image.count != 4 or image.shape != (height, width)
                    or str(image.crs) != report['crs'] or image.res != (10.0, 10.0)
                    or image.nodata != -9999 or image.dtypes != ('float32',) * 4
                    or list(image.transform)[:6] != report['transform']):
                raise ValueError('Reflectance grid/type mismatch.')
            arrays = {}
            for name in ['scene_classes', 'usable_mask', 'study_mask']:
                with rasterio.open(folder / f'{name}.tif') as layer:
                    if (layer.crs != image.crs or layer.transform != image.transform
                            or layer.shape != image.shape or layer.count != 1
                            or layer.dtypes != ('uint8',)):
                        raise ValueError(f'Layer grid/type mismatch: {name}')
                    arrays[name] = layer.read(1)
            if any(not np.isin(arrays[n], [0, 1]).all() for n in ['usable_mask', 'study_mask']):
                raise ValueError('Masks must be binary.')
            valid, inside = arrays['usable_mask'].astype(bool), arrays['study_mask'].astype(bool)
            rows, cols = np.indices(image.shape)
            x = image.transform.c + (cols + .5) * image.transform.a
            y = image.transform.f + (rows + .5) * image.transform.e
            lon, lat = project_points(image.crs, 'EPSG:4326', x.ravel(), y.ravel())
            lon, lat = np.array(lon).reshape(image.shape), np.array(lat).reshape(image.shape)
            west, south, east, north = report['bbox_lon_lat']
            expected_inside = (lon >= west) & (lon <= east) & (lat >= south) & (lat <= north)
            if not np.array_equal(inside, expected_inside):
                raise ValueError('Study mask differs from research box pixel centers.')
            if (int(inside.sum()) != study or int(valid.sum()) != usable
                    or (valid & ~inside).any()
                    or not np.isin(arrays['scene_classes'][valid], [4, 5, 6]).all()):
                raise ValueError('Quality/coverage mask mismatch.')
            for band in image.read():
                if (not np.isfinite(band[valid]).all()
                        or not (band[~valid] == image.nodata).all()
                        or (band[valid] == image.nodata).any()):
                    raise ValueError('Reflectance validity mismatch.')
            classes, counts = np.unique(arrays['scene_classes'][inside], return_counts=True)
            histogram = {str(int(k)): int(v) for k, v in zip(classes, counts)}
            if histogram != report['scene_class_counts_inside_study']:
                raise ValueError('SCL histogram differs from raster.')
    return {
        'checks': 'checksums, provenance, coverage' + (', saved raster grids and masks' if rasters else ''),
        'scene_id': report['scene_id'], 'study_pixels': study, 'usable_pixels': usable,
        'usable_fraction': report['usable_fraction'],
        'scope': 'Provisional research crop only; not forest labels or beat totals.',
    }



print(json.dumps(verify(OUT, rasters=True),indent=2))
with zipfile.ZipFile(OUT.parent/'forestguard_sample.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in SAMPLE_FILES + ['checksums.json']:
        archive.write(OUT/name,arcname=name)
print('Verified export:', OUT.parent/'forestguard_sample.zip')
print('ZIP bytes:', (OUT.parent/'forestguard_sample.zip').stat().st_size)
